In [1]:
# Table S1: the full 72-gene chaperone/protease census, with the inclusion rule
# and a flag for whether each gene also appears in the 61-gene high-confidence
# ATFS-1 regulon. The census itself is frozen (data/chaperone_protease_census.csv)
# and was already independently reproduced from the raw annotation in
# census_build.ipynb; this notebook re-validates that reproduction here rather than
# trusting the frozen file blindly, then formats it as a supplementary table.
import os
if os.path.basename(os.getcwd()) == "scripts":
    os.chdir("..")

import gzip
import pandas as pd

FROZEN = "data/chaperone_protease_census.csv"
census = pd.read_csv(FROZEN)
if len(census) != 72:
    raise RuntimeError(f"Census has {len(census)} rows, expected 72.")
role_counts = census["role"].value_counts()
if role_counts.get("chaperone", 0) != 65 or role_counts.get("protease", 0) != 7:
    raise RuntimeError(f"Role split changed: {role_counts.to_dict()}, expected 65 chaperone / 7 protease.")
print(f"Census loaded: {len(census)} genes ({role_counts['chaperone']} chaperone, {role_counts['protease']} protease)")

Census loaded: 72 genes (65 chaperone, 7 protease)


In [2]:
# Independent reproduction check - the same rebuild census_build.ipynb performs,
# repeated here rather than assumed still true, since Table S1 is what a reviewer
# would actually check numbers against.
GFF = "data/raw/c_elegans.PRJNA13758.WS285.protein_annotation.gff3.gz"

SINGLE_DOMAIN = {
    "PF00012": "HSP70",       "PF00226": "DnaJ",
    "PF00183": "HSP90",       "PF00011": "HSP20",
    "PF00118": "Cpn60_TCP1",  "PF01920": "Prefoldin",
    "PF00574": "CLP_protease", "PF01434": "Peptidase_M41",
}
LON_DOMAINS = {"PF05362": "Lon_C", "PF02190": "LON_substr_bdg"}
PROTEASE_DOMAINS = {"PF00574", "PF01434", "PF05362", "PF02190"}
ALL_DOMAINS = set(SINGLE_DOMAIN) | set(LON_DOMAINS)

protein_gene, protein_domains, has_signal_peptide = {}, {}, set()
with gzip.open(GFF, "rt") as fh:
    for line in fh:
        if line.startswith("#"):
            continue
        fields = line.rstrip("\n").split("\t")
        if len(fields) < 9:
            continue
        protein, source, feature, attrs = fields[0], fields[1], fields[2], fields[8]
        if source == "WormBase" and feature == "CDS":
            a = dict(kv.split("=", 1) for kv in attrs.split(";") if "=" in kv)
            if "wormbase_geneid" in a:
                protein_gene[protein] = (a["wormbase_geneid"], a.get("wormbase_genename", ""))
        elif source == "Pfam" and feature == "motif":
            for pfam in ALL_DOMAINS:
                if pfam in attrs:
                    protein_domains.setdefault(protein, set()).add(pfam)
        elif source == "SignalP" and feature == "signal_peptide":
            has_signal_peptide.add(protein)

qualifying, er_targeted = {}, set()
for protein, domains in protein_domains.items():
    if protein not in protein_gene:
        continue
    matched = domains & set(SINGLE_DOMAIN)
    if set(LON_DOMAINS) <= domains:
        matched |= set(LON_DOMAINS)
    if not matched:
        continue
    gene_id, gene_name = protein_gene[protein]
    if protein in has_signal_peptide:
        er_targeted.add(gene_id)
    entry = qualifying.setdefault(gene_id, {"name": gene_name, "domains": set()})
    entry["domains"] |= matched

rebuilt_census = {g: e for g, e in qualifying.items() if g not in er_targeted}
MANUAL_EXCLUSIONS = {"ppk-3", "lido-17", "rme-8"}
rebuilt_census = {g: e for g, e in rebuilt_census.items() if e["name"] not in MANUAL_EXCLUSIONS}

name_to_gene = {}
for protein, (gene_id, gene_name) in protein_gene.items():
    if gene_name:
        name_to_gene.setdefault(gene_name, gene_id)
for subunit in ("pfd-3", "pfd-5"):
    gene_id = name_to_gene[subunit]
    rebuilt_census[gene_id] = {"name": subunit, "domains": {"PF01920"}}

if set(rebuilt_census) != set(census["gene_id"]):
    raise RuntimeError("Rebuilt census gene membership no longer matches the frozen file.")
print(f"Reproduction check: {len(rebuilt_census)} genes, exact membership match against the frozen file.")

Reproduction check: 72 genes, exact membership match against the frozen file.


In [3]:
# Cross-reference: which of the 72 census genes also appear in the 61-gene
# high-confidence regulon (already computed and validated in table_s2.ipynb).
regulon = pd.read_csv("results/regulon_61.csv")
regulon_names = set(regulon["public_name"]) | set(regulon["seqname"])
census["in_61_regulon"] = census["public_name"].isin(regulon_names) | census["seqname"].isin(regulon_names)

n_in_regulon = int(census["in_61_regulon"].sum())
in_regulon_names = sorted(census.loc[census["in_61_regulon"], "public_name"])
if n_in_regulon != 2 or in_regulon_names != ["dnj-10", "ymel-1"]:
    raise RuntimeError(f"Expected exactly dnj-10 and ymel-1 in the regulon, got {in_regulon_names}.")
print(f"{n_in_regulon} of 72 census genes are in the 61-gene regulon: {in_regulon_names}")
print("(matches Analysis B's strict count exactly - this is the same fact from the other direction.)")

2 of 72 census genes are in the 61-gene regulon: ['dnj-10', 'ymel-1']
(matches Analysis B's strict count exactly - this is the same fact from the other direction.)


In [4]:
# Assemble and write. Column order: identity, role, domain evidence, inclusion
# note (manual additions carry their own explanatory text, everything else is
# domain-only), regulon membership flag.
table_s1 = census.rename(columns={"gene_id": "wbgene"})[
    ["wbgene", "seqname", "public_name", "role", "pfam_families", "in_61_regulon"]
].sort_values("public_name").reset_index(drop=True)

INCLUSION_RULE = (
    "Inclusion rule: membership decided from Pfam protein-domain annotations, not gene "
    "name. Eight domain families qualify a gene on their own (HSP70, HSP20, HSP90, DnaJ, "
    "Cpn60_TCP1, Prefoldin, CLP_protease, Peptidase_M41); the Lon protease additionally "
    "requires both Lon_C and LON_substr_bdg together, since either alone is not sufficient. "
    "Genes with a signal peptide co-occurring with a qualifying domain on the same protein "
    "isoform are excluded as ER-targeted. Two manual exclusions (ppk-3, rme-8) remove genes "
    "where the matched domain is a minor accessory feature rather than the protein's "
    "function. Two manual additions (pfd-3, pfd-5) restore canonical prefoldin subunits that "
    "carry no Pfam hit in this WormBase release but are unambiguous complex members."
)

os.makedirs("tables", exist_ok=True)
table_s1.to_csv("results/table_s1_census.csv", index=False)
table_s1.to_csv("tables/table_s1_census.csv", index=False)
with open("tables/table_s1_census_note.txt", "w") as fh:
    fh.write(INCLUSION_RULE + "\n")

print(f"Wrote {len(table_s1)} rows to results/table_s1_census.csv and tables/table_s1_census.csv")
print(f"Wrote inclusion-rule note to tables/table_s1_census_note.txt")
print(f"\n{table_s1['role'].value_counts().to_dict()}, "
      f"{int(table_s1['in_61_regulon'].sum())} in the 61-gene regulon")
print(table_s1.head(6).to_string(index=False))

Wrote 72 rows to results/table_s1_census.csv and tables/table_s1_census.csv
Wrote inclusion-rule note to tables/table_s1_census_note.txt

{'chaperone': 65, 'protease': 7}, 2 in the 61-gene regulon
        wbgene  seqname public_name      role pfam_families  in_61_regulon
WBGene00008591  F08H9.3     F08H9.3 chaperone         HSP20          False
WBGene00008592  F08H9.4     F08H9.4 chaperone         HSP20          False
WBGene00008714  F11F1.1     F11F1.1 chaperone         HSP70          False
WBGene00009691  F44E5.4     F44E5.4 chaperone         HSP70          False
WBGene00009692  F44E5.5     F44E5.5 chaperone         HSP70          False
WBGene00010640 K07F5.16    K07F5.16 chaperone          DnaJ          False
